## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到所在的代码树根 (`solutions/` 或 `tutorials/`)，
   这样 `from attention.mha import ...` 这种导入能直接生效。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd to the tree root (whichever of `solutions/` or `tutorials/` this notebook lives in), turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys

# Walk up from the notebook's CWD until we find a directory named
# `solutions` or `tutorials`. Works no matter which tree the student
# opened. 不论 notebook 位于 solutions/ 还是 tutorials/ 都能正确定位。
ROOTS = {'solutions', 'tutorials'}
if os.path.basename(os.getcwd()) not in ROOTS:
    while os.path.basename(os.getcwd()) not in ROOTS and os.getcwd() != '/':
        os.chdir('..')
    if os.path.basename(os.getcwd()) not in ROOTS:
        # Fallback: maybe we were started at the repo root.
        if os.path.isdir('tutorials'):
            os.chdir('tutorials')
        elif os.path.isdir('solutions'):
            os.chdir('solutions')

assert os.path.basename(os.getcwd()) in ROOTS, (
    f'could not locate solutions/ or tutorials/ from {os.getcwd()}')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the attention chapter's reference .pt files live
control_folder = 'attention/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 1 章 · Attention

## 这一章你要学什么

AlphaFold 3 整个模型可以看作**一堆 attention 块以不同方式排列组合**:

- **Pairformer** 里有 *triangle attention* (沿一对残基的两条 starting/ending  轴) 与 *standard attention with pair bias*
- **DiffusionTransformer** 里有 *AttentionPairBias* (用 pair 张量作为偏置)，  并按 token 级 / atom 级两种粒度运行
- **ConfidenceHead** 内部又跑了一份小型 Pairformer + 四个分类头

这些 attention 长得花样多，但**它们底下都在跑同一段 scaled dot-product 数学**: 

$$\mathrm{Attention}(Q, K, V, b) = \mathrm{softmax}\Big(\frac{QK^\top}{\sqrt{d}} + b\Big)\, V$$

差别只是怎么算 $Q, K, V, b$、按哪些轴展开、要不要加 sigmoid 门控。本章把这套基础设施**从最底层的线性层一路搭到 AF3 标志性的 AttentionPairBias**，下面的Pairformer / Diffusion / Confidence 章节就只剩 "用这些零件按论文 Algorithm 接线"。

## 本章模块速览

| 文件 | 类 / 函数 | 角色 |
|---|---|---|
| `linear.py` | `Linear`, `LinearNoBias`, `BiasInitLinear` | 可选不同初始化策略的线性层 |
| `layer_norm.py` | `OpenFoldLayerNorm` | 参数名与 Protenix 融合算子对齐的 LN |
| `mha.py` | `_attention`, `Attention` | 缩放点积数学 + 多头封装 |
| `transition.py` | `AdaptiveLayerNorm`, `Transition` | FiLM 风格调制 (Alg 26) + SwiGLU FFN (Alg 11) |
| `attention_pair_bias.py` | `AttentionPairBias` | AF3 算法 24，主干最高频的复合块 |

## 测试是怎么工作的

每个 cell 调用 `test_module_shape` / `test_module_forward` 之前，测试 harness 会把你的模块参数**暂时**替换成 `torch.linspace(-1, 1, numel)`。这样:

- **数值是确定的**：参考解和你的实现见到同样的权重 → 任何对的实现都得到同样的输出。
- **不依赖训练好的权重**：每章测试都能离线、几毫秒内跑完。
- **形状对了不代表实现对了**：测试会先比 shape (`*_param_shapes.pt`)，再比  forward 输出 (`torch.allclose` 到 `atol=1e-6`)。

如果你跑测试得到 `output 'out' is None` 这种友好错误，说明 forward 还停留在 `pass` 没填；如果是 `appears to be uninitialized`，说明 `__init__` 没调 `super().__init__()`。

## 论文 + 实现的对照

本章每个组件都和 AF3 Supplementary 的某条 Algorithm 一一对应:

| 组件 | 论文 Alg | 论文页 (Abramson et al. 2024) |
|---|---|---|
| Linear / LinearNoBias / BiasInitLinear | (实现细节，非算法) | — |
| LayerNorm | (实现细节) | — |
| `_attention` 数学 | 见 attention is all you need | 经典 |
| `Attention` 模块 | (类于 OpenFold/AF2) | 经典 |
| `AdaptiveLayerNorm` | **Algorithm 26** | Suppl. p27 |
| `Transition` (SwiGLU) | **Algorithm 11** | Suppl. p21 |
| `AttentionPairBias` | **Algorithm 24** | Suppl. p27 |

Algorithm 编号在每节标题里都写明了。建议把 AF3 Supplementary PDF 打开放在一边对照看。

## 1.1 Linear (自定义初始化的线性层)

`torch.nn.Linear` 把一切都用默认 Kaiming 初始化。AF3 不接受这个 ——它对**不同角色的线性层用不同初始化**。

### 为什么初始化选择重要 — 一点信号传播分析

考虑一层线性变换 $y = W x$，输入 $x \in \mathbb{R}^{n_\text{in}}$ 服从均值 0 方差 1 的分布、$W_{ij}$ 独立同分布且均值 0。则:

$$\mathrm{Var}(y_j) = \sum_i W_{ij}^2 \, \mathrm{Var}(x_i) = n_\text{in} \cdot \mathrm{Var}(W) \cdot 1$$

要让前向方差**保持为 1**，需要 $\mathrm{Var}(W) = 1 / n_\text{in}$ —— 这就是 **fan-in 初始化**。

如果 W 之后紧跟 ReLU，**激活会砍掉一半**，方差减半，于是要把 $\mathrm{Var}(W)$ 加倍 → He 初始化 (scale=2)。SiLU/GELU 砍得没有 ReLU 那么硬，但实践中也用 He 风格。

### AF3 的四种策略

| `initializer` | 用于哪里 | 数学 | 直观 |
|---|---|---|---|
| `"default"` | 普通线性层 | $\mathrm{Var}(W) = 1/n_\text{in}$, 截断正态 | 保前向方差 |
| `"relu"` | ReLU/SiLU 前的层 (Transition 的 a/b 分支) | $\mathrm{Var}(W) = 2/n_\text{in}$ | 补偿激活方差损失 |
| `"zeros"` | 残差块的最末投影 | $W = 0$ | 起手贡献 0，残差恒等 |
| (隐含) | sigmoid 门控源 | 截断正态 (用于 OpenFold-style "gating" init) | sigmoid(z) 起手 ≈ 0.5 |

`bias` 一律零初始化 (即使 weight 非零)。

### Linear.forward 的两条路径

AF3 主干一般跑 bf16 节省显存，但有些位置 (坐标投影、噪声水平条件) **必须 fp32**:

- bf16 mantissa 只有 7 bit，坐标差只有几埃米时无法分辨 0.01 Å 的细节 → 配位错位
- 扩散噪声水平 `t / sigma_data` 在 σ 接近 0 时是个很小的数，bf16 直接归零 → 网络永远看不到清晰信号

所以 `Linear` 提供 `precision` 参数: 给定时就 disable autocast + 临时把 input/weight/bias 上提到该 dtype，算完再回到原 dtype。这样混合精度训练时只有这几条危险路径走 fp32，其余照样 bf16，速度 / 精度兼得。

**任务**: 打开 `attention/linear.py` 把 `Linear._init_params` 和 `Linear.forward` 两个 TODO 块填好。每个 TODO 上方都有详细伪代码 + 中英双语说明。这一格只测 `default` 策略 + 普通 forward 路径，1.3 节会单独检查 `BiasInitLinear` 的常数偏置初始化。

In [ ]:
from attention.linear import Linear
from attention.control_values.attention_checks import (
    c_a, c_z, test_module_shape, test_module_forward, test_inputs,
)

lin = Linear(in_features=c_a, out_features=c_z)
test_module_shape(lin, 'linear_default', control_folder)
test_module_forward(lin, 'linear_default',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)
print('Linear (default init) ✓')

## 1.2 LinearNoBias (无 bias 的 Linear)

Pairformer / Diffusion 主干里**大部分线性层都不带 bias** —— LayerNorm 已经提供了平移，再叠 bias 多余而且会让某些初始化策略 (如零初始化) 出现对称性问题。所以仓库里你会看到很多 `LinearNoBias` 而不是 `Linear`。

这里没有新代码: `LinearNoBias = partial(Linear, bias=False)` —— 它就是`Linear` 固定 `bias=False`。**只要 1.1 你的 `Linear` 写对了，这格自动通过**。我们单独跑测试既是冗余检查，也提醒你「无 bias 的 Linear」是 AF3 默认形态。

In [ ]:
from attention.linear import LinearNoBias

lin_nb = LinearNoBias(in_features=c_a, out_features=c_z)
test_module_shape(lin_nb, 'linear_nobias', control_folder)
test_module_forward(lin_nb, 'linear_nobias',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)
print('LinearNoBias ✓')

## 1.3 BiasInitLinear (起手输出常数的 Linear)

AdaLN-Zero ([Peebles & Xie 2023, DiT](https://arxiv.org/abs/2212.09748)) 是 AF3**深层 Transformer 能稳训练**的关键技巧之一。它要求 attention / FFN 的输出门是一个 sigmoid，并且**初始几乎关闭**，让每个 block 起手对残差是近零贡献:

$$x \leftarrow x + \sigma(W_s s + b_s) \cdot \mathrm{Block}(x)$$

### 为什么不能让门初始在 0.5

如果让 $W_s$ / $b_s$ 都走默认初始化，sigmoid 在 0.5 附近 —— 每个 block 都给残差贡献约 0.5·Block(x)，于是连续 L 层后 $x$ 的方差 ≈ $L \cdot 0.25 \cdot \mathrm{Var}(\text{Block})$。AF3 主干 48 个 PairformerBlock + 24 个 DiffusionTransformerBlock，如果每个都贡献 0.5 倍的 noise，**激活方差会指数级爆炸**，需要极小的 learning rate 才能稳。

### 为什么是 biasinit = -2

$$\sigma(-2) = \frac{1}{1 + e^2} \approx 0.119$$

也就是说 block 起手只贡献 12% 而非 50% 的 update。L=48 层后总方差只放大到约$48 \cdot 0.119^2 \cdot \mathrm{Var}(\text{Block}) \approx 0.68$，远远稳定 ——网络能从这个状态出发，慢慢学习"该让哪些 block 开门"。

AF3 用 -2，DiT 原论文用 -3 (更保守)。在 AF3 这个量级足够稳定，也保留了"门能向上学到 0.5+"的余量。

### 为什么 weight 还要零初始化

光让 bias = -2 让 sigmoid 起手在 0.12 是不够的 —— 如果 $W_s$ 非零，那么 sigmoid 的输入就成了 $W_s s - 2$，**取决于 $s$ 的实际值**: $s$ 大时门可能仍开得很大。零初始化让**门完全只由 bias 控制**: $\sigma(b) = \sigma(-2)$ 与 $s$ 无关。训练才学到怎么让 $W_s$ 起作用。

### 代码 3 步

`BiasInitLinear` 就是承担这个角色的 Linear 变体: 继承 `Linear`，覆盖 `__init__`:

1. `super().__init__(...)` — 走父类的 Kaiming 初始化
2. `nn.init.zeros_(self.weight)` — 覆盖 weight 为 0
3. `nn.init.constant_(self.bias, biasinit)` — bias 设为 -2 (默认) 或调用方传入

**任务**: 在 `attention/linear.py` 填 `BiasInitLinear.__init__` 的 TODO 块。

In [ ]:
from attention.linear import BiasInitLinear
from attention.control_values.attention_checks import c_s

bil = BiasInitLinear(in_features=c_s, out_features=c_a,
                     bias=True, biasinit=-2.0)
test_module_shape(bil, 'bias_init_linear', control_folder)
test_module_forward(bil, 'bias_init_linear',
                    inputs=(test_inputs['x_s'],),
                    output_names='out',
                    control_folder=control_folder)
print('BiasInitLinear ✓')

## 1.4 LayerNorm (可选 scale / offset)

PyTorch 自带的 `nn.LayerNorm` 给定 `c_in` 后默认带 scale ($\gamma$) 和 offset($\beta$):

$$\mathrm{LN}(x)_i = \gamma_i \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta_i$$

但 AF3 有两个地方需要**关闭** scale 或 offset:

1. **AdaLN-Zero (Algorithm 26)** 内部对 `a` 做 LN 时**两个都关**: scale / offset 改由   `s` 经过线性层生成 —— 标准的 FiLM 调制。如果再带原生 $\gamma$ / $\beta$，   两套调制混在一起容易学到 trivial 解。
2. **DiffusionConditioning 里的 z / s 归一化**只关 offset，保留 scale。

所以我们自己写一个 `OpenFoldLayerNorm`，构造时通过 `create_scale` / `create_offset`决定要不要建可学参数。**关键**: 关掉某个参数时，仍要在 `state_dict` 里给它留一个 `None` 位 (`register_parameter(name, None)`)，否则 Protenix 权重加载会失败。

另外 forward 里给 **bf16** 输入做了个特殊路径: autocast off + 临时把 weight / bias降到 bf16，这样得到的输出与上游融合算子位级一致。

**任务**: 打开 `attention/layer_norm.py` 填两个 TODO 块。

In [ ]:
from pairformer.triangle_ops import LayerNorm

ln = LayerNorm(c_a)
test_module_shape(ln, 'layer_norm', control_folder)
test_module_forward(ln, 'layer_norm',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)
print('LayerNorm ✓')

## 1.5 `_attention` (核心点积数学)

整个 AF3 attention 体系都在调一个无可学参数的纯函数:

$$\mathrm{out} = \mathrm{softmax}\Big(\frac{Q K^\top}{\sqrt{d}} + b\Big) \, V$$

### 为什么是 $1/\sqrt{d}$ 而不是 $1/d$ 或 $1$

假设 $Q, K$ 各分量独立同分布、均值 0 方差 1。点积 $Q_i \cdot K_j = \sum_k Q_{ik} K_{jk}$ 是$d$ 个独立项之和，因此:

$$\mathbb{E}[Q_i \cdot K_j] = 0, \quad \mathrm{Var}(Q_i \cdot K_j) = d$$

注意标准差是 $\sqrt{d}$。如果不缩放、d 大 (例如 64)，attention logits 的方差就是 64，softmax 输入有相当部分 $|z| > 8$；这导致 softmax 输出近似 one-hot，**几乎所有梯度都死掉**:

$$\frac{\partial \, \mathrm{softmax}_i}{\partial z_j} = \mathrm{softmax}_i \cdot (\delta_{ij} - \mathrm{softmax}_j)$$

当 $\mathrm{softmax}_i \to 1$ 或 $\to 0$，导数趋近 0 ——「饱和的 softmax」。

缩放 $1/\sqrt{d}$ 让 logits 方差回到 1，softmax 工作在线性区，梯度健康。这是 [Attention Is All You Need](https://arxiv.org/abs/1706.03762) 第 3.2.1 节的核心观察。

### 为什么我们提前缩放 Q 而不是在 `_attention` 内部缩放

理论上等价: $\frac{Q K^\top}{\sqrt{d}} = (\frac{Q}{\sqrt{d}}) K^\top$。但实现上有差:

- **提前缩放 Q (我们的做法)**: matmul 算 $\tilde{Q} K^\top$，传给 `F.scaled_dot_product_attention` 时 `scale=1.0`。
- **运行时缩放**: matmul 完再除一次，多一次 elementwise op。CUDA 上不是瓶颈但 CPU 上能省 5-10%。

在 `_attention(q, k, v, attn_bias, use_efficient_implementation, inplace_safe)` 中:

- `q, k, v` 都已经被外面拆好头维 + 缩放，shape 是 `[..., H, T_q, d]` 和   `[..., H, T_kv, d]`。**调用前 Q 已乘 1/sqrt(d)**，所以函数内部传 `scale=1.0`。
- `attn_bias` 是 broadcastable 的 `[..., H, T_q, T_kv]`，包含 mask 和  pair / triangle bias 的总和。

**两条路径**:

1. `use_efficient_implementation=True`: 调 `F.scaled_dot_product_attention`，   PyTorch 在 CUDA 上能融合 + 走 FlashAttention，速度最快；缺点是要求 Q/K/V   dtype 一致 (我们测试时显式 disable 走显式路径)。
2. 显式数学路径: `Q @ K^T + bias → softmax → · V`。在 `autocast("cuda", enabled=False)`   下跑、Q/K 升 fp32 算 attention，再把 weight 强转回原 dtype 与 V 相乘 ——   避免 bf16 / fp16 下 softmax 精度灾难。

**任务**: 在 `attention/mha.py` 文件顶部 (类 `Attention` 之前) 找到 `_attention` 函数，把整个函数体 (上面 TODO 块) 实现出来。这是后续所有 attention 模块共同调用的底座。

In [ ]:
from attention.mha import _attention

out = _attention(
    test_inputs['q_raw'].double(),
    test_inputs['k_raw'].double(),
    test_inputs['v_raw'].double(),
    attn_bias=None,
    use_efficient_implementation=False,
)
expected = torch.load(f'{control_folder}/attention_function_out.pt')
assert torch.allclose(out, expected), '_attention output mismatch'
print('_attention ✓')

## 1.6 Attention (多头注意力模块)

现在把 `_attention` 包成一个**带参数、带门控、能搬到 GPU 上跑批量数据**的 `nn.Module`。拆解成 3 个小方法 + 1 个 forward，更易读也好测:

**`__init__`** 创建 5 个线性层 (命名要与 Protenix state_dict 严格一致):

- `linear_q`, `linear_k`, `linear_v`: 把输入 `c_q` / `c_k` / `c_v` 各自投到  `H * c_hidden`。Q 是否带 bias 由 `q_linear_bias` 控制 (AF3 默认带)；K/V 不带。
- `linear_o`: 输出投影 `H * c_hidden → c_q`，无 bias，**可选 zero-init** —— 当外层  没有 adaLN-Zero 门时 (`zero_init=True`) 让 attention block 起手为 0。
- `linear_g` (可选): sigmoid 门控源，**zero-init** + `sigmoid` 一起作用让门起手在 0.5。

**`_prep_qkv`** 做 4 件事 (按这个顺序最稳):

1. 三个线性投影 → `[*, T, H * c_hidden]`
2. `view(... + (H, c_hidden))` 拆出头维 → `[*, T, H, c_hidden]`
3. `transpose(-2, -3)` 把头维提前 → `[*, H, T, c_hidden]` (这样 `_attention` 沿 T 求和)
4. **预先**对 Q 乘 `1/sqrt(c_hidden)` —— 这样 `_attention` 内部 `scale=1.0`

**`_wrap_up`**: 三步 —— 可选门控、展平头维、`linear_o` 投回 `c_q`。

**`forward`** 是调度逻辑: 给了 `n_queries`/`n_keys` 就走局部窗口路径(`global_attention_with_bias` 或 `local_cross_attention`)，否则走全连接路径。局部 attention 是 AtomTransformer 用的 —— 让原子级注意力 O(N) 而非 O(N²)。

**任务**: 把 `Attention.__init__` / `_prep_qkv` / `_wrap_up` / `forward` 四个 TODO 全部填好。

In [ ]:
from attention.mha import Attention
from attention.control_values.attention_checks import N_head, c_hidden

attn = Attention(
    c_q=c_a, c_k=c_a, c_v=c_a,
    c_hidden=c_hidden, num_heads=N_head,
    gating=True, q_linear_bias=True,
    use_efficient_implementation=False,
    zero_init=False,
)
test_module_shape(attn, 'mha_gated', control_folder)

from attention.control_values.attention_checks import test_module_method
test_module_method(
    attn, 'mha_gated',
    inputs=(test_inputs['q_x'], test_inputs['kv_x'], test_inputs['attn_bias']),
    output_names='out',
    control_folder=control_folder,
    method=lambda q_x, kv_x, b: attn(q_x=q_x, kv_x=kv_x, attn_bias=b),
)
print('Attention ✓')

## 1.7 AdaptiveLayerNorm + Transition

两个看上去无关、但都在 `attention/transition.py` 的高频组件:

### `AdaptiveLayerNorm` (Algorithm 26) — FiLM 的演化版

FiLM (Feature-wise Linear Modulation, [Perez et al. 2017](https://arxiv.org/abs/1709.07871))提出了一个简单想法: **给一个条件 c，用它生成 affine 变换 (γ, β)，按 channel 改主流 x**:

$$\mathrm{FiLM}(x, c) = \gamma(c) \odot x + \beta(c)$$

原版 FiLM 直接乘 γ。AF3 / DiT 把 γ 换成 sigmoid 之后再乘 —— 这是 AdaLN-Zero 的关键变体:

$$\mathrm{AdaLN}(a, s) = \sigma(W_g \, \mathrm{LN}_s(s)) \cdot \mathrm{LN}_a(a) + W_b \, \mathrm{LN}_s(s)$$

为什么用 sigmoid 而不是直接的 γ?

- γ 没有上下界，初始一旦偏大会瞬间放大主流方差。sigmoid 自然 clamp 到 (0, 1)，方差不会爆炸。
- 配合 zero-init 让 sigmoid 起手在 0.5：主流被「半衰」缩放、shift 完全 0 —— 起手等价于  $a \leftarrow 0.5 \cdot \mathrm{LN}(a)$，深层堆叠下不爆炸 (参 1.3 节的方差分析)。

**关键设计**:

- `LN_a` **关掉 scale + offset**: 调制完全由 s 控制。如果再叠原生 γ/β，参数化冗余、训练歧义。
- `LN_s` **保留 scale**: 让 s 进入门控前先做一次稳定归一化。
- `linear_s` (sigmoid 门源) + `linear_nobias_s` (shift 源) 都 **zero-init**: 起手只有  sigmoid 的 0.5 倍主流，shift 严格为 0。

### `Transition` (Algorithm 11) — 为什么 SwiGLU 而不是普通 MLP

Vanilla Transformer 的 FFN 是 $W_2 \,\text{GELU}(W_1 x)$。AF3 / Gemini / Llama 普遍换成 SwiGLU([Shazeer 2020](https://arxiv.org/abs/2002.05202)):

$$\mathrm{Transition}(x) = W_o \, (\mathrm{SiLU}(W_a \, \mathrm{LN}(x)) \odot W_b \, \mathrm{LN}(x))$$

区别在中间多一路 "value" 分支 $W_b x$，与 SiLU 门做 element-wise 乘:

- **更强表达**: GLU 风格门控让 FFN 学到「按通道选择性放大 / 关闭」，普通 MLP 做不到。
- **同算力下性能更好**: 论文实测同 FLOP 比 GELU MLP 在 LM 任务低 ~0.4 PPL。
- **代价**: 参数量从 $2 \cdot c \cdot n c$ 变成 $3 \cdot c \cdot n c$。AF3 把 n 从 4 调到 2/4，  保持总参数量接近的同时享受 SwiGLU 的能力。

**实现要点**:

- 两路扩宽线性层 `linear_no_bias_a` / `linear_no_bias_b` 都用 `"relu"` 初始化  (fan-in 截断正态 scale=2)，因为后面跟 SiLU/逐元素乘。
- 输出投影 `linear_no_bias` 用 `"zeros"` 初始化，残差起手为 0。
- `n` 通常 2 或 4: AF3 主干 `c_a=384` 时 transition 隐藏维到 1536 (n=4)；
  diffusion 的 `ConditionedTransitionBlock` 用 n=2 节省 FLOPs。

**任务**: 在 `attention/transition.py` 填:

- `AdaptiveLayerNorm.__init__` (4 个子模块) + `forward` (LN + 调制公式)
- `Transition.__init__` (LN + 3 linears) + `forward` (LN + SwiGLU + out)

In [ ]:
from attention.transition import AdaptiveLayerNorm, Transition
from attention.control_values.attention_checks import n_factor

adaln = AdaptiveLayerNorm(c_a=c_a, c_s=c_s)
test_module_shape(adaln, 'adaptive_layer_norm', control_folder)
test_module_forward(adaln, 'adaptive_layer_norm',
                    inputs=(test_inputs['x_a'], test_inputs['x_s']),
                    output_names='out',
                    control_folder=control_folder)

tr = Transition(c_in=c_a, n=n_factor)
test_module_shape(tr, 'transition', control_folder)
test_module_forward(tr, 'transition',
                    inputs=(test_inputs['x_a'],),
                    output_names='out',
                    control_folder=control_folder)
print('AdaLN + Transition ✓')

## 1.8 AttentionPairBias (AF3 算法 24)

本章的**顶点**。这一个组件出现在:

- **PairformerBlock** 的单序列 update (用 pair `z` 作为偏置 bias 调整每对 token 的注意力)
- **DiffusionTransformerBlock** 的核心 attention 分支 (`has_s=True` 启用 AdaLN-Zero 门)
- **AtomTransformer** 的局部窗口版 (走 `local_multihead_attention` 路径)
- **ConfidenceHead** 内部的小 Pairformer

把这块写对 = AF3 主干基本就接通了。

### Forward 流程

```text
    a ─── AdaLN(s) or LN ──┐
                            ├── attention(q=a, kv=a, bias = Linear(LN(z)))
                            │       │
                            │       └── 局部窗口路径 (n_queries/n_keys 给了时)
                            │           或全连接路径
                            │
                            └── × sigmoid(linear_a_last(s))    ← adaLN-Zero output gate
                                       (起手 ≈ sigmoid(-2) = 0.12，初闭)
    最后返回累计 update，由调用方做残差加。
```

### Pair bias 怎么来

$$b_{ij}^h = (\mathrm{Linear}_{c_z \to H} \, \mathrm{LN}(z))_{ij}^h$$

投出的 H 通道恰好作为多头注意力的每头 bias，加到 $Q K^\top$ 上。几何意义: 让 token i 在关注 token j 时，看 pair 表示 $z_{ij}$ 决定多 / 少关注。

### `local_*` vs `standard_*`

- `standard_multihead_attention(q, kv, z)`: pair `z` 是 `[..., N, N, c_z]`，  PairformerBlock 用这条。
- `local_multihead_attention(q, kv, z, n_queries, n_keys)`: pair `z` 已经被  rearrange 成 `[..., n_blocks, n_queries, n_keys, c_z]` (dense-trunk 形)，  AtomTransformer 用这条避免 O(N²) 显存。

**任务**: 在 `attention/attention_pair_bias.py` 填四个 TODO ——`__init__`、两个 multihead 辅助、和顶层 `forward`。

In [ ]:
from attention.attention_pair_bias import AttentionPairBias

apb = AttentionPairBias(
    has_s=True, create_offset_ln_z=False,
    n_heads=N_head, c_a=c_a, c_s=c_s, c_z=c_z,
    biasinit=-2.0, cross_attention_mode=False,
)
# Disable the SDP fast path so the test runs in double precision.
apb.attention.use_efficient_implementation = False

test_module_shape(apb, 'attention_pair_bias', control_folder)
test_module_method(
    apb, 'attention_pair_bias',
    inputs=(test_inputs['x_a'], test_inputs['x_s'], test_inputs['x_z']),
    output_names='out',
    control_folder=control_folder,
    method=lambda a, s, z: apb(a=a, s=s, z=z),
)
print('AttentionPairBias ✓')

## 章节小结

完成本章后，你已经手写了:

1. **`Linear` / `LinearNoBias` / `BiasInitLinear`** —— 带初始化策略选择的线性层 + adaLN-Zero 门的特殊变体；
2. **`OpenFoldLayerNorm`** —— 可选 scale / offset 的 LN，bf16 走融合算子兼容路径；
3. **`_attention`** —— 缩放点积 + softmax + 加权和的纯函数，两条 dtype 策略；
4. **`Attention`** —— 包成模块的多头注意力，含 5 个线性层、可选 sigmoid 门、局部 / 全连接两条 forward 路径；
5. **`AdaptiveLayerNorm`** —— FiLM 风格的条件归一化 (Algorithm 26)；
6. **`Transition`** —— SwiGLU FFN (Algorithm 11)；
7. **`AttentionPairBias`** —— AF3 主干最高频的复合块 (Algorithm 24)。

**下一站**: 这些零件马上会在第 2 章 Pairformer 里被反复组合 ——你会发现 TriangleAttention 内部用的 mha、PairformerBlock 用的 AttentionPairBias、OuterProductMean 用的 LayerNorm，全是本章交付的成品。